# Reto 9: Sistema de Gestión de Calificaciones

## Programación para Ciencia de Datos
### Instituto Politécnico Nacional
### Febrero - Julio 2026

In [1]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from datetime import datetime
import json

## PARTE 1: Carga y Exploración de Datos

In [2]:
def cargar_datos() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Carga los datos de estudiantes, calificaciones y materias.
    Retorna: (df_estudiantes, df_calificaciones, df_materias)
    """
    estudiantes = pd.DataFrame({
        'boleta': ['2021630001','2021630002','2021630003','2021630004','2021630005',
                   '2022630001','2022630002','2022630003','2022630004','2022630005',
                   '2023630001','2023630002','2023630003','2023630004','2023630005'],
        'nombre': ['Juan Pérez García','María López Ruiz','Pedro Sánchez Torres',
                   'Ana Martínez Díaz','Luis Rodríguez Vega','Carmen Flores Luna',
                   'Roberto Díaz Mora','Laura Torres Silva','Diego Ramírez Cruz',
                   'Sofía Vargas Romo','Carlos Mendoza Ríos','Patricia Ortiz León',
                   'Miguel Ángel Castro','Fernanda Reyes Paz','Andrés Guzmán Villa'],
        'semestre': [4,4,4,4,4,3,3,3,3,3,2,2,2,2,2],
        'carrera': ['CD']*15,
        'email': ['juan.perez@ipn.mx','maria.lopez@ipn.mx','pedro.sanchez@ipn.mx',
                  'ana.martinez@ipn.mx','luis.rodriguez@ipn.mx','carmen.flores@ipn.mx',
                  'roberto.diaz@ipn.mx','laura.torres@ipn.mx','diego.ramirez@ipn.mx',
                  'sofia.vargas@ipn.mx','carlos.mendoza@ipn.mx','patricia.ortiz@ipn.mx',
                  'miguel.castro@ipn.mx','fernanda.reyes@ipn.mx','andres.guzman@ipn.mx']
    })
    materias = pd.DataFrame({
        'materia_id': ['MAT101','MAT102','PROG101','PROG102','EST101','EST102','BD101'],
        'nombre': ['Cálculo Diferencial','Cálculo Integral','Programación I',
                   'Programación II','Probabilidad','Estadística Inferencial','Bases de Datos'],
        'creditos': [8,8,6,6,6,6,6],
        'semestre_materia': [1,2,1,2,2,3,3]
    })
    np.random.seed(42)
    data = []
    for boleta in estudiantes['boleta']:
        sem = estudiantes[estudiantes['boleta']==boleta]['semestre'].values[0]
        for mat in materias[materias['semestre_materia']<=sem]['materia_id']:
            base = np.random.uniform(5,10)
            p1 = round(min(10,max(0,base+np.random.normal(0,1))),1)
            p2 = round(min(10,max(0,base+np.random.normal(0,1))),1)
            fi = round(min(10,max(0,base+np.random.normal(0,0.5))),1)
            if np.random.random()<0.05: p2 = np.nan
            data.append({'boleta':boleta,'materia_id':mat,'parcial_1':p1,'parcial_2':p2,'final':fi})
    return estudiantes, pd.DataFrame(data), materias

In [3]:
def info_general(df_estudiantes: pd.DataFrame, df_calificaciones: pd.DataFrame) -> Dict:
    """Genera información general del sistema."""
    return {
        "total_estudiantes": len(df_estudiantes),
        "total_registros_calif": len(df_calificaciones),
        "semestres": sorted(df_estudiantes['semestre'].unique().tolist()),
        "materias_con_registros": df_calificaciones['materia_id'].nunique()
    }

In [4]:
def validar_datos(df_calificaciones: pd.DataFrame) -> Dict:
    """Valida la integridad de los datos."""
    cols = ['parcial_1','parcial_2','final']
    nulos = int(df_calificaciones[cols].isna().any(axis=1).sum())
    fuera = sum(int(((df_calificaciones[c].dropna()<0)|(df_calificaciones[c].dropna()>10)).sum()) for c in cols)
    return {"registros_con_nulos": nulos, "calificaciones_fuera_rango": fuera, "datos_validos": nulos==0 and fuera==0}

## PARTE 2: Consultas y Filtros

In [5]:
def buscar_estudiante(df_estudiantes: pd.DataFrame, criterio: str, valor: str) -> pd.DataFrame:
    """Busca estudiantes por criterio: 'boleta', 'nombre' (parcial) o 'semestre'."""
    if criterio == 'nombre':
        return df_estudiantes[df_estudiantes['nombre'].str.contains(valor,case=False,na=False)].reset_index(drop=True)
    elif criterio == 'boleta':
        return df_estudiantes[df_estudiantes['boleta']==valor].reset_index(drop=True)
    elif criterio == 'semestre':
        return df_estudiantes[df_estudiantes['semestre']==int(valor)].reset_index(drop=True)
    else:
        raise ValueError(f"Criterio '{criterio}' no válido. Usa: 'boleta', 'nombre' o 'semestre'.")

In [6]:
def obtener_kardex(boleta: str, df_estudiantes: pd.DataFrame,
                   df_calificaciones: pd.DataFrame, df_materias: pd.DataFrame) -> Dict:
    """Obtiene el kardex completo de un estudiante."""
    est = df_estudiantes[df_estudiantes['boleta']==boleta]
    if est.empty:
        return {k:None for k in ['estudiante','materias','promedio_general','creditos_cursados','materias_aprobadas','materias_reprobadas']}
    c = df_calificaciones[df_calificaciones['boleta']==boleta].copy()
    c = c.merge(df_materias[['materia_id','nombre','creditos']], on='materia_id', how='left')
    c['promedio'] = c[['parcial_1','parcial_2','final']].mean(axis=1).round(2)
    c['estatus'] = c['promedio'].apply(lambda x: 'Aprobada' if x >= 6 else 'Reprobada')
    return {
        "estudiante": est.iloc[0].to_dict(),
        "materias": c[['materia_id','nombre','parcial_1','parcial_2','final','promedio','estatus','creditos']].reset_index(drop=True),
        "promedio_general": round(c['promedio'].mean(), 2),
        "creditos_cursados": int(c['creditos'].sum()),
        "materias_aprobadas": int((c['estatus']=='Aprobada').sum()),
        "materias_reprobadas": int((c['estatus']=='Reprobada').sum())
    }

In [7]:
def filtrar_por_rendimiento(df_calificaciones: pd.DataFrame, df_estudiantes: pd.DataFrame,
                            min_promedio: float = None, max_promedio: float = None) -> pd.DataFrame:
    """Filtra estudiantes por rango de promedio general."""
    d = df_calificaciones.copy()
    d['pm'] = d[['parcial_1','parcial_2','final']].mean(axis=1)
    p = d.groupby('boleta')['pm'].mean().round(2).reset_index()
    p.columns = ['boleta','promedio_general']
    if min_promedio is not None: p = p[p['promedio_general'] >= min_promedio]
    if max_promedio is not None: p = p[p['promedio_general'] <= max_promedio]
    return p.merge(df_estudiantes, on='boleta', how='left').sort_values('promedio_general', ascending=False).reset_index(drop=True)

## PARTE 3: Cálculos y Estadísticas

In [8]:
def calcular_promedio_materia(df_calificaciones: pd.DataFrame, materia_id: str) -> Dict:
    """Calcula estadísticas de una materia específica."""
    d = df_calificaciones[df_calificaciones['materia_id']==materia_id].copy()
    if d.empty: return {"materia": materia_id, "error": "No se encontraron registros"}
    d['pa'] = d[['parcial_1','parcial_2','final']].mean(axis=1)
    return {
        "materia": materia_id, "inscritos": len(d),
        "promedio_parcial1": round(d['parcial_1'].mean(), 2),
        "promedio_parcial2": round(d['parcial_2'].mean(), 2),
        "promedio_final": round(d['final'].mean(), 2),
        "promedio_general": round(d['pa'].mean(), 2),
        "tasa_aprobacion": round((d['pa']>=6).sum()/len(d)*100, 1),
        "calificacion_maxima": d['pa'].max(),
        "calificacion_minima": d['pa'].min()
    }

In [9]:
def ranking_estudiantes(df_calificaciones: pd.DataFrame, df_estudiantes: pd.DataFrame,
                        top_n: int = 10) -> pd.DataFrame:
    """Genera ranking de mejores estudiantes por promedio general."""
    d = df_calificaciones.copy()
    d['pm'] = d[['parcial_1','parcial_2','final']].mean(axis=1)
    p = d.groupby('boleta')['pm'].mean().round(2).reset_index()
    p.columns = ['boleta','promedio_general']
    p = p.sort_values('promedio_general', ascending=False).head(top_n)
    p['posicion'] = range(1, len(p)+1)
    r = p.merge(df_estudiantes[['boleta','nombre','semestre']], on='boleta', how='left')
    return r[['posicion','boleta','nombre','semestre','promedio_general']].reset_index(drop=True)

In [10]:
def estadisticas_por_semestre(df_estudiantes: pd.DataFrame,
                              df_calificaciones: pd.DataFrame) -> pd.DataFrame:
    """Calcula estadísticas agrupadas por semestre."""
    d = df_calificaciones.copy()
    d['pm'] = d[['parcial_1','parcial_2','final']].mean(axis=1)
    d['ap'] = d['pm'] >= 6
    pe = d.groupby('boleta').agg(pg=('pm','mean'), ta=('ap','mean')).reset_index()
    pe = pe.merge(df_estudiantes[['boleta','semestre']], on='boleta', how='left')
    r = pe.groupby('semestre').agg(
        Estudiantes=('boleta','count'), Promedio=('pg','mean'),
        Tasa_Aprob=('ta','mean'), Mejor_Promedio=('pg','max'), Peor_Promedio=('pg','min')
    ).round(2)
    r['Tasa_Aprob'] = (r['Tasa_Aprob']*100).round(1)
    return r

## PARTE 4: Identificación de Riesgo y Reportes

In [11]:
def identificar_estudiantes_riesgo(df_calificaciones: pd.DataFrame,
                                   df_estudiantes: pd.DataFrame,
                                   umbral_promedio: float = 7.0,
                                   max_reprobadas: int = 2) -> pd.DataFrame:
    """Identifica estudiantes en riesgo académico."""
    d = df_calificaciones.copy()
    d['pm'] = d[['parcial_1','parcial_2','final']].mean(axis=1)
    d['rep'] = d['pm'] < 6
    r = d.groupby('boleta').agg(pg=('pm','mean'), rep=('rep','sum')).reset_index()
    r['pg'] = r['pg'].round(2); r['rep'] = r['rep'].astype(int)
    e = r[(r['pg'] < umbral_promedio) | (r['rep'] > max_reprobadas)].copy()
    def motivo(row):
        b = row['pg'] < umbral_promedio; rr = row['rep'] > max_reprobadas
        return 'Ambos' if b and rr else ('Bajo promedio' if b else 'Mat. reprobadas')
    e['motivo'] = e.apply(motivo, axis=1)
    e = e.merge(df_estudiantes[['boleta','nombre','semestre']], on='boleta', how='left')
    return e[['boleta','nombre','semestre','pg','rep','motivo']].sort_values('pg').reset_index(drop=True)

In [12]:
def generar_reporte_academico(df_estudiantes: pd.DataFrame,
                              df_calificaciones: pd.DataFrame,
                              df_materias: pd.DataFrame) -> Dict:
    """Genera reporte académico completo."""
    d = df_calificaciones.copy()
    d['pm'] = d[['parcial_1','parcial_2','final']].mean(axis=1)
    d['ap'] = d['pm'] >= 6
    pm_mat = d.groupby('materia_id').agg(inscritos=('boleta','count'),promedio=('pm','mean'),tasa=('ap','mean')).round(2).reset_index()
    pm_mat['tasa'] = (pm_mat['tasa']*100).round(1)
    pm_mat = pm_mat.merge(df_materias[['materia_id','nombre']], on='materia_id', how='left')
    return {
        "resumen_general": {"total_estudiantes": len(df_estudiantes),
                            "promedio_global": round(d['pm'].mean(), 2),
                            "tasa_aprobacion": round(d['ap'].mean()*100, 1)},
        "por_semestre": estadisticas_por_semestre(df_estudiantes, df_calificaciones),
        "por_materia": pm_mat,
        "mejores_estudiantes": ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=5),
        "estudiantes_riesgo": identificar_estudiantes_riesgo(df_calificaciones, df_estudiantes),
        "fecha_generacion": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

In [13]:
def exportar_kardex(boleta: str, kardex: Dict, formato: str = 'csv') -> str:
    """Exporta el kardex de un estudiante a CSV o JSON."""
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"kardex_{boleta}_{ts}.{formato}"
    if formato == 'csv':
        df_ex = kardex['materias'].copy()
        res = pd.DataFrame([{'materia_id':'RESUMEN','nombre':'','parcial_1':'','parcial_2':'','final':'',
            'promedio':kardex['promedio_general'],
            'estatus':f"Aprobadas:{kardex['materias_aprobadas']} Reprobadas:{kardex['materias_reprobadas']}",
            'creditos':kardex['creditos_cursados']}])
        pd.concat([df_ex,res],ignore_index=True).to_csv(fname, index=False, encoding='utf-8-sig')
    elif formato == 'json':
        datos = {"estudiante":kardex['estudiante'],"promedio_general":kardex['promedio_general'],
                 "creditos_cursados":kardex['creditos_cursados'],"materias_aprobadas":kardex['materias_aprobadas'],
                 "materias_reprobadas":kardex['materias_reprobadas'],
                 "materias":kardex['materias'].to_dict(orient='records')}
        with open(fname,'w',encoding='utf-8') as f: json.dump(datos,f,ensure_ascii=False,indent=2)
    else:
        raise ValueError(f"Formato '{formato}' no soportado. Usa 'csv' o 'json'.")
    print(f"✅ Kardex exportado: {fname}")
    return fname

## Funciones de Visualización

In [14]:
def mostrar_kardex(kardex: Dict) -> None:
    """Muestra el kardex de forma legible."""
    if kardex['estudiante'] is None:
        print("❌ Estudiante no encontrado"); return
    est = kardex['estudiante']
    print("="*70); print("                         KARDEX ACADÉMICO"); print("="*70)
    print(f"\n📋 DATOS DEL ESTUDIANTE"); print("-"*40)
    print(f"Boleta:   {est.get('boleta','N/A')}"); print(f"Nombre:   {est.get('nombre','N/A')}")
    print(f"Semestre: {est.get('semestre','N/A')}"); print(f"Carrera:  {est.get('carrera','N/A')}")
    print(f"Email:    {est.get('email','N/A')}")
    print(f"\n📚 CALIFICACIONES"); print("-"*70)
    if kardex['materias'] is not None and len(kardex['materias'])>0:
        print(kardex['materias'].to_string(index=False))
    else: print("Sin calificaciones registradas")
    print(f"\n📊 RESUMEN"); print("-"*40)
    print(f"Promedio General:    {kardex.get('promedio_general',0):.2f}")
    print(f"Créditos Cursados:   {kardex.get('creditos_cursados',0)}")
    print(f"Materias Aprobadas:  {kardex.get('materias_aprobadas',0)}")
    print(f"Materias Reprobadas: {kardex.get('materias_reprobadas',0)}"); print("="*70)

In [15]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte académico completo."""
    print("="*70); print("              REPORTE ACADÉMICO - CIENCIA DE DATOS")
    print(f"              Generado: {reporte['fecha_generacion']}"); print("="*70)
    res = reporte.get('resumen_general',{})
    print(f"\n📊 RESUMEN GENERAL"); print("-"*40)
    print(f"Total de estudiantes: {res.get('total_estudiantes','N/A')}")
    print(f"Promedio global:      {res.get('promedio_global',0):.2f}")
    print(f"Tasa de aprobación:   {res.get('tasa_aprobacion',0):.1f}%")
    if reporte.get('por_semestre') is not None:
        print(f"\n📅 ESTADÍSTICAS POR SEMESTRE"); print("-"*60)
        print(reporte['por_semestre'].to_string())
    if reporte.get('mejores_estudiantes') is not None:
        print(f"\n🏆 TOP 5 ESTUDIANTES"); print("-"*60)
        print(reporte['mejores_estudiantes'].to_string(index=False))
    if reporte.get('estudiantes_riesgo') is not None and len(reporte['estudiantes_riesgo'])>0:
        print(f"\n⚠️  ESTUDIANTES EN RIESGO ({len(reporte['estudiantes_riesgo'])})"); print("-"*60)
        print(reporte['estudiantes_riesgo'].to_string(index=False))
    else: print(f"\n✅ No hay estudiantes en riesgo académico")
    print("\n"+"="*70)

## BONUS: Funcionalidades Extra

In [16]:
def predecir_riesgo_proximo_semestre(df_calificaciones: pd.DataFrame,
                                     df_estudiantes: pd.DataFrame) -> pd.DataFrame:
    """
    Predice estudiantes en riesgo el próximo semestre.
    Criterios: tendencia decreciente (parcial_2 < parcial_1) y final < parcial_1.
    """
    d = df_calificaciones.dropna(subset=['parcial_1','parcial_2','final']).copy()
    d['tend_dec'] = d['parcial_2'] < d['parcial_1']
    d['final_menor'] = d['final'] < d['parcial_1']
    d['senal'] = d['tend_dec'] & d['final_menor']
    r = d.groupby('boleta').agg(total=('materia_id','count'), senales=('senal','sum')).reset_index()
    r['pct_riesgo'] = (r['senales']/r['total']*100).round(1)
    e = r[r['pct_riesgo']>50].merge(df_estudiantes[['boleta','nombre','semestre']], on='boleta', how='left')
    return e[['boleta','nombre','semestre','senales','total','pct_riesgo']].sort_values('pct_riesgo',ascending=False).reset_index(drop=True)

In [17]:
def comparar_estudiantes(boleta1: str, boleta2: str,
                         df_calificaciones: pd.DataFrame,
                         df_estudiantes: pd.DataFrame,
                         df_materias: pd.DataFrame) -> Dict:
    """Compara el rendimiento de dos estudiantes."""
    k1 = obtener_kardex(boleta1, df_estudiantes, df_calificaciones, df_materias)
    k2 = obtener_kardex(boleta2, df_estudiantes, df_calificaciones, df_materias)
    if k1['materias'] is not None and k2['materias'] is not None:
        com = set(k1['materias']['materia_id']) & set(k2['materias']['materia_id'])
        m1 = k1['materias'][k1['materias']['materia_id'].isin(com)][['materia_id','nombre','promedio']]
        m2 = k2['materias'][k2['materias']['materia_id'].isin(com)][['materia_id','promedio']]
        comp = m1.merge(m2, on='materia_id', suffixes=('_est1','_est2'))
        comp['diferencia'] = (comp['promedio_est1']-comp['promedio_est2']).round(2)
    else: comp = pd.DataFrame()
    g = k1['estudiante'].get('nombre') if (k1['promedio_general'] or 0)>=(k2['promedio_general'] or 0) else k2['estudiante'].get('nombre')
    return {"estudiante_1":k1['estudiante'],"estudiante_2":k2['estudiante'],
            "promedio_1":k1['promedio_general'],"promedio_2":k2['promedio_general'],
            "ganador":g,"materias_comunes":comp}

## Prueba de la Implementación

In [18]:
# Cargar datos
df_estudiantes, df_calificaciones, df_materias = cargar_datos()

print('DATOS CARGADOS')
print('='*50)
print(f'\nEstudiantes ({len(df_estudiantes)} registros):')
print(df_estudiantes.head().to_string())
print(f'\nCalificaciones ({len(df_calificaciones)} registros):')
print(df_calificaciones.head().to_string())
print(f'\nMaterias ({len(df_materias)} registros):')
print(df_materias.to_string())

DATOS CARGADOS

Estudiantes (15 registros):
       boleta                nombre  semestre carrera                  email
0  2021630001     Juan Pérez García         4      CD      juan.perez@ipn.mx
1  2021630002      María López Ruiz         4      CD     maria.lopez@ipn.mx
2  2021630003  Pedro Sánchez Torres         4      CD   pedro.sanchez@ipn.mx
3  2021630004     Ana Martínez Díaz         4      CD    ana.martinez@ipn.mx
4  2021630005   Luis Rodríguez Vega         4      CD  luis.rodriguez@ipn.mx

Calificaciones (95 registros):
       boleta materia_id  parcial_1  parcial_2  final
0  2021630001     MAT101        5.8        7.2    7.0
1  2021630001     MAT102        6.1        4.5    4.8
2  2021630001    PROG101        3.9        7.5    6.9
3  2021630001    PROG102        4.9        5.8    5.4
4  2021630001     EST101        8.6        6.4    6.1

Materias (7 registros):
  materia_id                   nombre  creditos  semestre_materia
0     MAT101      Cálculo Diferencial         8

In [19]:
print('\nINFORMACIÓN GENERAL')
print('='*50)
info = info_general(df_estudiantes, df_calificaciones)
for k, v in info.items():
    print(f'  {k}: {v}')

INFORMACIÓN GENERAL
  total_estudiantes: 15
  total_registros_calif: 95
  semestres: [2, 3, 4]
  materias_con_registros: 7


In [20]:
print('\nVALIDACIÓN DE DATOS')
print('='*50)
validacion = validar_datos(df_calificaciones)
for k, v in validacion.items():
    print(f'  {k}: {v}')

VALIDACIÓN DE DATOS
  registros_con_nulos: 5
  calificaciones_fuera_rango: 0
  datos_validos: False


In [21]:
print('\nBÚSQUEDA DE ESTUDIANTES')
print('='*50)
print("\n-- Buscar por nombre 'María' --")
print(buscar_estudiante(df_estudiantes, 'nombre', 'María').to_string())
print("\n-- Buscar por semestre 3 --")
print(buscar_estudiante(df_estudiantes, 'semestre', '3').to_string())
print("\n-- Buscar por boleta '2022630001' --")
print(buscar_estudiante(df_estudiantes, 'boleta', '2022630001').to_string())

BÚSQUEDA DE ESTUDIANTES

-- Buscar por nombre 'María' --
       boleta            nombre  semestre carrera               email
0  2021630002  María López Ruiz         4      CD  maria.lopez@ipn.mx

-- Buscar por semestre 3 --
       boleta              nombre  semestre carrera                 email
0  2022630001  Carmen Flores Luna         3      CD  carmen.flores@ipn.mx
1  2022630002   Roberto Díaz Mora         3      CD   roberto.diaz@ipn.mx
2  2022630003  Laura Torres Silva         3      CD   laura.torres@ipn.mx
3  2022630004  Diego Ramírez Cruz         3      CD  diego.ramirez@ipn.mx
4  2022630005   Sofía Vargas Romo         3      CD   sofia.vargas@ipn.mx

-- Buscar por boleta '2022630001' --
       boleta              nombre  semestre carrera                 email
0  2022630001  Carmen Flores Luna         3      CD  carmen.flores@ipn.mx


In [22]:
print('\nKARDEX DE ESTUDIANTE')
kardex = obtener_kardex('2021630001', df_estudiantes, df_calificaciones, df_materias)
mostrar_kardex(kardex)

                         KARDEX ACADÉMICO

📋 DATOS DEL ESTUDIANTE
----------------------------------------
Boleta:   2021630001
Nombre:   Juan Pérez García
Semestre: 4
Carrera:  CD
Email:    juan.perez@ipn.mx

📚 CALIFICACIONES
----------------------------------------------------------------------
materia_id                  nombre  parcial_1  parcial_2  final  promedio   estatus  creditos
    MAT101     Cálculo Diferencial        5.8        7.2    7.0      6.67  Aprobada         8
    MAT102        Cálculo Integral        6.1        4.5    4.8      5.13 Reprobada         8
   PROG101          Programación I        3.9        7.5    6.9      6.10  Aprobada         6
   PROG102         Programación II        4.9        5.8    5.4      5.37 Reprobada         6
    EST101            Probabilidad        8.6        6.4    6.1      7.03  Aprobada         6
    EST102 Estadística Inferencial        4.8        4.7    5.8      5.10 Reprobada         6
     BD101          Bases de Datos        7.

In [23]:
print('\nFILTRAR POR RENDIMIENTO (promedio >= 8.0)')
print('='*50)
print(filtrar_por_rendimiento(df_calificaciones, df_estudiantes, min_promedio=8.0).to_string())

FILTRAR POR RENDIMIENTO (promedio >= 8.0)
       boleta  promedio_general               nombre  semestre carrera                  email
0  2023630003              8.91  Miguel Ángel Castro         2      CD   miguel.castro@ipn.mx
1  2023630004              8.59   Fernanda Reyes Paz         2      CD  fernanda.reyes@ipn.mx
2  2021630005              8.44  Luis Rodríguez Vega         4      CD  luis.rodriguez@ipn.mx
3  2023630005              8.43  Andrés Guzmán Villa         2      CD   andres.guzman@ipn.mx


In [24]:
print('\nESTADÍSTICAS DE MATERIA: MAT101')
print('='*50)
stats = calcular_promedio_materia(df_calificaciones, 'MAT101')
for k, v in stats.items():
    print(f'  {k}: {v}')

ESTADÍSTICAS DE MATERIA: MAT101
  materia: MAT101
  inscritos: 15
  promedio_parcial1: 7.76
  promedio_parcial2: 7.91
  promedio_final: 7.82
  promedio_general: 7.79
  tasa_aprobacion: 100.0
  calificacion_maxima: 9.6
  calificacion_minima: 6.0


In [25]:
print('\nRANKING TOP 5 ESTUDIANTES')
print('='*50)
print(ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=5).to_string(index=False))

RANKING TOP 5 ESTUDIANTES
 posicion     boleta              nombre  semestre  promedio_general
        1 2023630003 Miguel Ángel Castro         2              8.91
        2 2023630004  Fernanda Reyes Paz         2              8.59
        3 2021630005 Luis Rodríguez Vega         4              8.44
        4 2023630005 Andrés Guzmán Villa         2              8.43
        5 2022630003  Laura Torres Silva         3              7.96


In [26]:
print('\nESTADÍSTICAS POR SEMESTRE')
print('='*50)
print(estadisticas_por_semestre(df_estudiantes, df_calificaciones).to_string())

ESTADÍSTICAS POR SEMESTRE
          Estudiantes  Promedio  Tasa_Aprob  Mejor_Promedio  Peor_Promedio
semestre                                                                  
2                   5      7.96        92.0            8.91           6.46
3                   5      7.75        91.0            7.96           7.41
4                   5      7.30        86.0            8.44           6.20


In [27]:
print('\nESTUDIANTES EN RIESGO')
print('='*50)
riesgo = identificar_estudiantes_riesgo(df_calificaciones, df_estudiantes)
print(riesgo.to_string(index=False))

ESTUDIANTES EN RIESGO
    boleta              nombre  semestre   pg  rep        motivo
2021630001   Juan Pérez García         4 6.20    3         Ambos
2023630002 Patricia Ortiz León         2 6.46    1 Bajo promedio
2021630002    María López Ruiz         4 6.86    0 Bajo promedio


In [28]:
# Reporte completo
reporte = generar_reporte_academico(df_estudiantes, df_calificaciones, df_materias)
mostrar_reporte(reporte)

              REPORTE ACADÉMICO - CIENCIA DE DATOS
              Generado: 2026-06-20 04:30:31

📊 RESUMEN GENERAL
----------------------------------------
Total de estudiantes: 15
Promedio global:      7.64
Tasa de aprobación:   89.5%

📅 ESTADÍSTICAS POR SEMESTRE
------------------------------------------------------------
          Estudiantes  Promedio  Tasa_Aprob  Mejor_Promedio  Peor_Promedio
semestre                                                                  
2                   5      7.96        92.0            8.91           6.46
3                   5      7.75        91.0            7.96           7.41
4                   5      7.30        86.0            8.44           6.20

🏆 TOP 5 ESTUDIANTES
------------------------------------------------------------
 posicion     boleta              nombre  semestre  promedio_general
        1 2023630003 Miguel Ángel Castro         2              8.91
        2 2023630004  Fernanda Reyes Paz         2              8.59
        3 2

In [29]:
# Exportar kardex en ambos formatos
archivo_csv = exportar_kardex('2021630001', kardex, formato='csv')
archivo_json = exportar_kardex('2021630001', kardex, formato='json')
print(f'Archivos generados: {archivo_csv}, {archivo_json}')

✅ Kardex exportado: kardex_2021630001_20260620_043031.csv
✅ Kardex exportado: kardex_2021630001_20260620_043031.json
Archivos generados: kardex_2021630001_*.csv, kardex_2021630001_*.json


In [30]:
# BONUS: Predicción de riesgo futuro
print('\nBONUS: PREDICCIÓN DE RIESGO PRÓXIMO SEMESTRE')
print('='*50)
prediccion = predecir_riesgo_proximo_semestre(df_calificaciones, df_estudiantes)
if len(prediccion) > 0:
    print(prediccion.to_string(index=False))
else:
    print('✅ Ningún estudiante muestra tendencia de riesgo')

BONUS: PREDICCIÓN DE RIESGO PRÓXIMO SEMESTRE
✅ Ningún estudiante muestra tendencia de riesgo


In [31]:
# BONUS: Comparar dos estudiantes
print('\nBONUS: COMPARACIÓN DE ESTUDIANTES')
print('='*50)
comp = comparar_estudiantes('2021630001', '2021630002', df_calificaciones, df_estudiantes, df_materias)
print(f"Estudiante 1: {comp['estudiante_1']['nombre']} → Promedio: {comp['promedio_1']}")
print(f"Estudiante 2: {comp['estudiante_2']['nombre']} → Promedio: {comp['promedio_2']}")
print(f"Mejor rendimiento: {comp['ganador']}")
print('\nMaterias en común:')
print(comp['materias_comunes'].to_string(index=False))

BONUS: COMPARACIÓN DE ESTUDIANTES
Estudiante 1: Juan Pérez García → Promedio: 6.2
Estudiante 2: María López Ruiz → Promedio: 6.86
Mejor rendimiento: María López Ruiz

Materias en común:
materia_id                  nombre  promedio_est1  promedio_est2  diferencia
    MAT101     Cálculo Diferencial           6.67           6.03        0.64
    MAT102        Cálculo Integral           5.13           7.93       -2.80
   PROG101          Programación I           6.10           8.03       -1.93
   PROG102         Programación II           5.37           6.37       -1.00
    EST101            Probabilidad           7.03           6.70        0.33
    EST102 Estadística Inferencial           5.10           6.80       -1.70
     BD101          Bases de Datos           7.97           6.17        1.80
